In [1]:

#単純にDBに接続してカラム数をカウントしてみる
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
PRAGMA table_info(stock_price)
"""
key_df =pd.read_sql(query,conn)
display(key_df)

,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0


In [59]:
import sqlite3
import pandas as pd

conn=sqlite3.connect("all_stocks.db")

query="""
SELECT COUNT(*) as cnt
FROM stock_price"""

pd.read_sql(query,conn)

,cnt
0,6934822


In [50]:
last_df.isna().sum()
last_df["date"].max()
last_df["ticker"].nunique()

3749

In [ ]:
import pandas as pd
import yfinance as yf

df=yf.download("7203.T",start="2025-01-01")

df.columns=df.columns.get_level_values(0)
df.columns.name = None
df.reset_index(inplace=True)
display(df.head(2))

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2025-01-06,2870.214600,2958.866196,2859.728927,2957.912953,41298500
1,2025-01-07,2909.297363,2980.790582,2854.485896,2863.541704,44700900


In [34]:
import pandas as pd
import sqlite3



conn= sqlite3.connect("all_stocks.db")
tickers="7203.T"
query = f"""SELECT MAX(date) as max_date FROM stock_price WHERE ticker='{tickers}'"""

last_date_df=pd.read_sql(query,conn)

last_date = last_date_df.loc[0, "max_date"]

if pd.isna(last_date):
    start_date = "2018-01-01"
else:
    start_date = (
    pd.to_datetime(last_date)
    + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")


df = yf.download(
        tickers,
        start=start_date,
        progress=False
        )

if df.empty:
    print("追加データなし")

else:
    print("新規データあり")
    display(df.head())

新規データあり


Price,Close,High,Low,Open,Volume
Ticker,7203.T,7203.T,7203.T,7203.T,7203.T
Date,,,,,
2026-04-13,3319.0,3337.0,3280.0,3295.0,14353200
2026-04-14,3326.0,3355.0,3288.0,3355.0,15920200
2026-04-15,3381.0,3386.0,3344.0,3355.0,20317000


In [ ]:
import sqlite3
import pandas as pd
import yfinance as yf
import time
DB_PATH = "all_stocks.db"

def update_stock(ticker):

    # DB接続
    conn = sqlite3.connect(DB_PATH)

    # 最新日付取得
    query = f"""
    SELECT MAX(date) as max_date
    FROM stock_price
    WHERE ticker = '{ticker}'
    """
    last_date_df = pd.read_sql(query, conn)
    last_date = last_date_df.loc[0, "max_date"]

    # 開始日決定
    if pd.isna(last_date):
        start_date = "2005-01-01"
    else:
        start_date = (
            pd.to_datetime(last_date) + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d")

    print(f"{ticker} | start: {start_date}")

    # データ取得
    df = yf.download(ticker, start=start_date, progress=False)

    if df.empty:
        print("追加データなし")
        conn.close()

        return

    # カラム整理
    df.columns = df.columns.get_level_values(0)
    df.columns.name = None

    df["ticker"] = ticker
    df = df.reset_index()

    # 🔥 日付フォーマット統一（超重要）
    df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")

    # カラム名統一
    df = df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume"
    })

    df = df[["ticker", "date", "open", "high", "low", "close", "volume"]]


     # =====================
        # ⑤ 重複除去（最重要）
     # =====================
    existing_dates = pd.read_sql(f"""
    SELECT date FROM stock_price WHERE ticker = '{ticker}'
    """, conn)

    df = df[~df["date"].isin(existing_dates["date"])]

    if df.empty:
        print(f"{ticker}: 追加なし（重複）")
        conn.close()
        return

    print(f"{ticker}: {len(df)}件追加")


    # DBに追加
    df.to_sql(
        "stock_price",
        conn,
        if_exists="append",
        index=False
    )

    print("DB更新完了:", ticker)

    conn.commit()
    conn.close()


# =====================
# メイン処理
# =====================
def main():

    conn = sqlite3.connect(DB_PATH)

    tickers_df = pd.read_sql("""
    SELECT コード
    FROM stock_master
    WHERE 市場・商品区分 LIKE '%プライム%'
    or 市場・商品区分 LIKE '%グロース%'
    or 市場・商品区分 LIKE '%スタンダード%'
    """, conn)

    conn.close()

    tickers = [f"{code}.T" for code in tickers_df["コード"].tolist()]
    print(f"[INFO] 対象銘柄数: {len(tickers)}")

    for i, ticker in enumerate(tickers[:10], 1):
        try:
            print(f"[{i}/{len(tickers)}] {ticker}")
            update_stock(ticker)
            time.sleep(0.3)
        except Exception as e:
            print(f"{ticker} 失敗:", e)

    with open("log.txt", "a") as f:
        f.write("実行完了\n")

if __name__ == "__main__":
    main()

[INFO] 対象銘柄数: 3773
[1/3773] 1301.T
1301.T | start: 2026-04-18
1301.T: 追加なし（重複）
[2/3773] 130A.T
130A.T | start: 2026-04-18
130A.T: 追加なし（重複）
[3/3773] 1332.T
1332.T | start: 2026-04-18
1332.T: 追加なし（重複）
[4/3773] 1333.T
1333.T | start: 2026-04-18
1333.T: 追加なし（重複）
[5/3773] 135A.T
135A.T | start: 2026-04-18
135A.T: 追加なし（重複）
[6/3773] 1375.T
1375.T | start: 2026-04-18
1375.T: 追加なし（重複）
[7/3773] 1376.T
1376.T | start: 2026-04-18
1376.T: 追加なし（重複）
[8/3773] 1377.T
1377.T | start: 2026-04-18
1377.T: 追加なし（重複）
[9/3773] 1379.T
1379.T | start: 2026-04-18
1379.T: 追加なし（重複）
[10/3773] 137A.T
137A.T | start: 2026-04-18
137A.T: 追加なし（重複）


In [1]:
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
PRAGMA table_info(stock_price)
"""
key_df =pd.read_sql(query,conn)
display(key_df)


,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0


In [5]:
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
DELETE FROM stock_price;
"""


conn.execute(query)
conn.commit()


df = pd.read_sql("SELECT COUNT(*) FROM stock_price", conn)
print(df)


key_df =pd.read_sql("""PRAGMA table_info(stock_price)""",conn)
display(key_df)



conn.close()

   COUNT(*)
0         0


,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0


In [14]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("all_stocks.db")

df = pd.read_sql("""
SELECT
    ticker,
    MIN(date) as start_date,
    MAX(date) as end_date,
    COUNT(*) as total_rows
FROM stock_price
GROUP BY ticker
""", conn)

print(df.head(20))
conn.close()

    ticker  start_date    end_date  total_rows
0   1301.T  2005-01-03  2026-04-21        5260
1   130A.T  2024-02-08  2026-04-21         537
2   1332.T  2005-01-03  2026-04-21        5260
3   1333.T  2014-04-01  2026-04-21        2968
4   135A.T  2024-02-22  2026-04-21         528
5   1375.T  2020-09-17  2026-04-21        1366
6   1376.T  2005-01-03  2026-04-21        5260
7   1377.T  2005-01-03  2026-04-21        5260
8   1379.T  2005-01-03  2026-04-21        5260
9   137A.T  2024-02-28  2026-04-21         525
10  1380.T  2005-01-03  2026-04-21        5260
11  1381.T  2005-01-03  2026-04-21        5260
12  1382.T  2005-08-01  2026-04-21        5110
13  1383.T  2011-11-29  2026-04-21        3542
14  1384.T  2015-02-20  2026-04-21        2750
15  138A.T  2024-02-29  2026-04-21         524
16  1401.T  2015-08-13  2026-04-21        2631
17  1407.T  2005-01-03  2026-04-21        5260
18  1414.T  2005-01-03  2026-04-21        5260
19  1417.T  2005-01-03  2026-04-21        5260


In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("all_stocks.db")

df = pd.read_sql("""
SELECT * from stock_master limit 15
""", conn)

print(df.head(20))
conn.close()

     コード                                   銘柄名     市場・商品区分
0   1301                                    極洋  プライム（内国株式）
1   1305                ｉＦｒｅｅＥＴＦ　ＴＯＰＩＸ（年１回決算型）     ETF・ETN
2   1306               ＮＥＸＴ　ＦＵＮＤＳ　ＴＯＰＩＸ連動型上場投信     ETF・ETN
3   1308                     上場インデックスファンドＴＯＰＩＸ     ETF・ETN
4   1309  ＮＥＸＴ　ＦＵＮＤＳ　ＣｈｉｎａＡＭＣ・中国株式・上証５０連動型上場投信     ETF・ETN
5   130A                     Ｖｅｒｉｔａｓ　Ｉｎ　Ｓｉｌｉｃｏ  グロース（内国株式）
6   1311       ＮＥＸＴ　ＦＵＮＤＳ　ＴＯＰＩＸ　Ｃｏｒｅ　３０連動型上場投信     ETF・ETN
7   1319           ＮＥＸＴ　ＦＵＮＤＳ　日経３００株価指数連動型上場投信     ETF・ETN
8   131A                               ＣＣＮグループ  PRO Market
9   1320                ｉＦｒｅｅＥＴＦ　日経２２５（年１回決算型）     ETF・ETN
10  1321               ＮＥＸＴ　ＦＵＮＤＳ　日経２２５連動型上場投信     ETF・ETN
11  1322    上場インデックスファンド中国Ａ株（パンダ）Ｅ　Ｆｕｎｄ　ＣＳＩ３００     ETF・ETN
12  1325       ＮＥＸＴ　ＦＵＮＤＳ　ブラジル株式指数・ボベスパ連動型上場投信     ETF・ETN
13  1326                          ＳＰＤＲゴールド・シェア     ETF・ETN
14  1328                 ＮＥＸＴ　ＦＵＮＤＳ　金価格連動型上場投信     ETF・ETN


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("all_stocks.db")

df = pd.read_sql("""
SELECT DISTINCT
    p.ticker,
    m.銘柄名
FROM stock_price p
LEFT JOIN stock_master m
ON REPLACE(p.ticker, '.T', '') = CAST(m.コード AS TEXT)
ORDER BY p.ticker
""", conn)

print(df.head(20))
conn.close()

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("all_stocks.db")

df = pd.read_sql("""
SELECT t.ticker, m.銘柄名
FROM (
    SELECT DISTINCT ticker
    FROM stock_price
) t
LEFT JOIN stock_master m
ON REPLACE(t.ticker, '.T', '') = CAST(m.コード AS TEXT)
ORDER BY t.ticker
""", conn)

print(df.head(20))
conn.close()

    ticker                銘柄名
0   1301.T                 極洋
1   130A.T  Ｖｅｒｉｔａｓ　Ｉｎ　Ｓｉｌｉｃｏ
2   1332.T               ニッスイ
3   1333.T             マルハニチロ
4   135A.T     ＶＲＡＩＮ　Ｓｏｌｕｔｉｏｎ
5   1375.T         ユキグニファクトリー
6   1376.T              カネコ種苗
7   1377.T             サカタのタネ
8   1379.T                ホクト
9   137A.T           Ｃｏｃｏｌｉｖｅ
10  1380.T               秋川牧園
11  1381.T              アクシーズ
12  1382.T                ホーブ
13  1383.T             ベルグアース
14  1384.T              ホクリヨウ
15  138A.T           光フードサービス
16  1401.T             エムビーエス
17  1407.T       ウエストホールディングス
18  1414.T     ショーボンドホールディングス
19  1417.T            ミライト・ワン


In [ ]:
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
del all_stocks.db
"""
key_df =pd.read_sql(query,conn)
display(key_df)


,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0


In [ ]:
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
copy all_stocks.db all_stocks_backup.db
"""
conn.execute(query)
conn.commit()


,seq,name,unique,origin,partial
0,0,sqlite_autoindex_stock_price_1,1,pk,0


In [7]:
#DB削除
import os

db_path = "all_stocks.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("削除完了")
else:
    print("ファイルなし")

削除完了


In [12]:
conn = sqlite3.connect("db/all_stocks.db")

df = pd.read_sql("""
SELECT name FROM sqlite_master WHERE type='table'
""", conn)

display(df)

df = pd.read_sql("SELECT * FROM stock_master LIMIT 5", conn)
display(df)


df = pd.read_sql("SELECT * FROM stock_price LIMIT 5", conn)
display(df)

,name
0,stock_price
1,stock_master


,ticker,company_name,market
0,1301.T,極洋,プライム（内国株式）
1,1305.T,ｉＦｒｅｅＥＴＦ ＴＯＰＩＸ（年１回決算型）,ETF・ETN
2,1306.T,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ連動型上場投信,ETF・ETN
3,1308.T,上場インデックスファンドＴＯＰＩＸ,ETF・ETN
4,1309.T,ＮＥＸＴ ＦＵＮＤＳ ＣｈｉｎａＡＭＣ・中国株式・上証５０連動型上場投信,ETF・ETN


,ticker,date,open,high,low,close,volume
